阶段1：构造模型

在票价确定的前提下，预测各航段的客流量（已完成）。

阶段2：解析过程

根据客流信息分配舱位，以确定总收益。
客舱分为两类：短途类（AB或BC，称为S类）和长途类（AC，称为L类），总容量为C。

根据是否满座进行分类：

若不满座（S+L≤C），则按S、L和C的预测比例分配舱位，以实现收益最大化。
若满座（S+L>C），则需根据销售量变化预测进行取舍，并作如下判断：
若票价AB+BC>AC，则优先将舱位拆分为AB和BC段销售，S的值取min(AB, BC)，其余舱位分配给L。
若票价AB+BC<AC，则优先分配舱位给AC段，剩余舱位分配给AB和BC段。

阶段3：求总收益最大值

以票价为自变量，总收益为因变量，绘制总收益曲线，并求出总收益的最大值。

In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import mean_squared_error
import joblib

# 加载数据
data = pd.read_csv('../../data-hh/my/202410221653-processed.csv', dtype={'aircraft': str})


## 数据预处理

In [2]:
data['flt_date'] = pd.to_datetime(data['flt_date'])
data['flt_date'] = (data['flt_date'] - data['flt_date'].min()).dt.days

label_fields = ['flt_no', 'a', 'b', 'c', 'aircraft']
label_encoders = {col: LabelEncoder() for col in label_fields}
for col in label_fields:
    data[col] = label_encoders[col].fit_transform(data[col])

X = data[['flt_date', 'flt_no', 'a', 'b', 'c', 'aircraft', 'ab_duration', 'bc_duration', 'ac_duration']]
y = data[['ab_pax', 'bc_pax', 'ac_pax']]



KeyError: 'flt_date'

## 训练模型

In [ ]:
# 分割数据
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 训练模型
ab_model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
bc_model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
ac_model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)

ab_model.fit(X_train, y_train['ab_pax'])
bc_model.fit(X_train, y_train['bc_pax'])
ac_model.fit(X_train, y_train['ac_pax'])



## 评估模型

In [ ]:
# 评估模型
ab_preds = ab_model.predict(X_test)
bc_preds = bc_model.predict(X_test)
ac_preds = ac_model.predict(X_test)

ab_mse = mean_squared_error(y_test['ab_pax'], ab_preds)
bc_mse = mean_squared_error(y_test['bc_pax'], bc_preds)
ac_mse = mean_squared_error(y_test['ac_pax'], ac_preds)

print(f"AB_PAX MSE: {ab_mse}")
print(f"BC_PAX MSE: {bc_mse}")
print(f"AC_PAX MSE: {ac_mse}")



## 获取预测结果

In [ ]:
# 获取测试集前 5 项数据
X_test_sample = X_test.head(5)
y_test_sample = y_test.head(5)

# 获取预测结果
ab_preds_sample = ab_model.predict(X_test_sample)
bc_preds_sample = bc_model.predict(X_test_sample)
ac_preds_sample = ac_model.predict(X_test_sample)

# 创建结果 DataFrame
results = pd.DataFrame({
    'AB_PAX_True': y_test_sample['ab_pax'].values,
    'AB_PAX_Pred': ab_preds_sample,
    'BC_PAX_True': y_test_sample['bc_pax'].values,
    'BC_PAX_Pred': bc_preds_sample,
    'AC_PAX_True': y_test_sample['ac_pax'].values,
    'AC_PAX_Pred': ac_preds_sample,
})

print("前 5 项预测结果与真实值对比：")
print(results)

## 保存模型

In [ ]:
# 保存模型
joblib.dump(ab_model, "ab_model.pkl")
joblib.dump(bc_model, "bc_model.pkl")
joblib.dump(ac_model, "ac_model.pkl")